In [1]:
import os
import pandas as pd
import xml.etree.ElementTree as ET
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt

In [2]:
DATASET_PATH = "/kaggle/input/bird-song"

ANNOTATION_TRAIN = os.path.join(DATASET_PATH, "Annotation/Training")
ANNOTATION_VAL = os.path.join(DATASET_PATH, "Annotation/Validation")

AUDIO_TRAIN = os.path.join(DATASET_PATH, "Audio/Training")
AUDIO_VAL = os.path.join(DATASET_PATH, "Audio/Validation")

OUTPUT_PATH = "/kaggle/working/"

In [3]:
species = [
 'Fringillidae_Serinus_canicollis',
 'Fringillidae_Serinus_serinus',
 'Paridae_Hypocnemis_cantator',
 'Paridae_Hypocnemis_hypoxantha',
 'Paridae_Hypocnemis_peruviana',
 'Paridae_Hypocnemis_striata',
 'Paridae_Saxicola_gutturalis',
 'Paridae_Saxicola_rubetra',
 'Paridae_Saxicola_rubicola',
 'Paridae_Saxicola_tectes',
 'Paridae_Saxicola_torquatus',
 'Troglodytidae_Troglodytes_aedon',
 'Troglodytidae_Troglodytes_hiemalis',
 'Troglodytidae_Troglodytes_pacificus',
 'Troglodytidae_Troglodytes_troglodytes',
 'Turdidae_Catharus_aurantiirostris',
 'Turdidae_Catharus_bicknelli',
 'Turdidae_Catharus_fuscater',
 'Turdidae_Catharus_fuscescens',
 'Turdidae_Catharus_guttatus',
 'Turdidae_Catharus_minimus',
 'Turdidae_Catharus_ustulatus'
]

LABEL_MAP = {
    'ae_Saxicola_rubicola':'Paridae_Saxicola_rubicola',
    'aridae_Saxicola_rubetra':'Paridae_Saxicola_rubetra',
    'idae_Saxicola_rubetra':'Paridae_Saxicola_rubetra',
    'roglodytidae_Troglodytes_troglodytes':'Troglodytidae_Troglodytes_troglodytes',
    'oglodytidae_Troglodytes_hiemalis':'Troglodytidae_Troglodytes_hiemalis',
    'Fringillidae_Serinus_icollis':'Fringillidae_Serinus_canicollis', 
}

In [4]:
svl_path = "/kaggle/input/bird-song/Annotation/Training"
print(os.listdir(svl_path)[:10])

example_svl = os.path.join(svl_path, os.listdir(svl_path)[0])
example_svl

['Turdidae_Catharus_fuscescens_United States_2017-06-10_XC628421_adul.svl', 'Troglodytidae_Troglodytes_pacificus_Canada_2019-06-14_XC481856_song.svl', 'Fringillidae_Serinus_serinus_Spain_2016-01-17_XC301937_song.svl', 'Turdidae_Catharus_ustulatus_Canada_2014-06-24_XC194219_song.svl', 'Paridae_Saxicola_rubetra_Sweden_2007-05-06_XC121169_song.svl', 'Turdidae_Catharus_ustulatus_United States_2016-06-07_XC322991_song.svl', 'Troglodytidae_Troglodytes_hiemalis_United States_2020-05-06_XC554701_song.svl', 'Paridae_Saxicola_rubetra_Sweden_1968-06-17_XC519960_adul.svl', 'Turdidae_Catharus_guttatus_United States_2008-04-19_XC83514_song.svl', 'Paridae_Hypocnemis_peruviana_Ecuador_2014-03-27_XC178582_song.svl']


'/kaggle/input/bird-song/Annotation/Training/Turdidae_Catharus_fuscescens_United States_2017-06-10_XC628421_adul.svl'

In [5]:
tree = ET.parse(example_svl)
root = tree.getroot()
for child in root:
    for elem in root.iter():
        print(elem.tag, elem.attrib)

sv {}
data {}
model {'id': '61', 'name': '', 'sampleRate': '48000', 'start': '87040', 'end': '1463296', 'type': 'sparse', 'dimensions': '2', 'resolution': '1', 'notifyOnAdd': 'true', 'dataset': '60', 'subtype': 'box', 'minimum': '1009.81', 'maximum': '5262.8', 'units': 'Hz'}
dataset {'id': '60', 'dimensions': '2'}
point {'frame': '87040', 'value': '1411.04', 'duration': '126976', 'extent': '3330.17', 'label': 'Turdidae_Catharus_fuscescens,9'}
point {'frame': '350208', 'value': '1772.14', 'duration': '116736', 'extent': '3490.66', 'label': 'Turdidae_Catharus_fuscescens,9'}
point {'frame': '603136', 'value': '1732.02', 'duration': '113664', 'extent': '3129.55', 'label': 'Turdidae_Catharus_fuscescens,9'}
point {'frame': '921600', 'value': '1651.77', 'duration': '107520', 'extent': '3129.55', 'label': 'Turdidae_Catharus_fuscescens,9'}
point {'frame': '1159168', 'value': '1451.16', 'duration': '105472', 'extent': '3249.92', 'label': 'Turdidae_Catharus_fuscescens,9'}
point {'frame': '1354752

In [6]:
sample_rate = 48000

def get_species(raw_label):
    """
    label = Family_genus_species_country,a number
    VD: Turdidae_Catharus_fuscescens,9
    """
    label = raw_label.strip()
    label_parts = label.split("_")
    if len(label_parts) >= 3:
        label = "_".join(label_parts[:3])
    else:
        label = "_".join(label_parts)
    label = label.split(",")[0]
    label = label.split(".")[0]
    label = label.strip()
    if label in LABEL_MAP:
        label = LABEL_MAP[label]
    return label
    
def parse_single_svl(svl_path, audio_filename, split, sample_rate=sample_rate):
    tree = ET.parse(svl_path)
    root = tree.getroot()
    
    rows = []
    for p in root.iter('point'):
        frame = int(p.attrib['frame'])
        duration = int(p.attrib['duration'])

        start_frame = frame
        end_frame = frame + duration

        start_time = start_frame / sample_rate
        end_time = end_frame / sample_rate

        value = float(p.attrib['value'])
        extent = float(p.attrib['extent'])

        f_low = max(0.0, value - extent / 2)
        f_high = value + extent / 2

        raw_label = p.attrib['label']
        species = get_species(raw_label)
        if species == '':
            continue

        rows.append({
            'audio_file': audio_filename,
            'split': split,
            'start_frame': start_frame,
            'end_frame': end_frame,
            'start_time': start_time,
            'end_time': end_time,
            'f_low': f_low,
            'f_high': f_high,
            'species': species
        })

    return pd.DataFrame(rows)

def resolve_audio_filename(svl_filename, audio_dir):
    base = os.path.splitext(svl_filename)[0]

    for fname in os.listdir(audio_dir):
        if os.path.splitext(fname)[0] == base:
            return fname
    return None

In [7]:
train_list_df = []

for svl_file in os.listdir(ANNOTATION_TRAIN):
    svl_path = os.path.join(ANNOTATION_TRAIN, svl_file)

    audio_file = resolve_audio_filename(svl_filename=svl_file, audio_dir=AUDIO_TRAIN)
    if audio_file is None:
        print(f"No audio for {svl_file}")
        continue

    df = parse_single_svl(
        svl_path=svl_path,
        audio_filename=audio_file,
        split="Training"
    )
    train_list_df.append(df)
train_df = pd.concat(train_list_df, ignore_index=True)
train_df.head()

,audio_file,split,start_frame,end_frame,start_time,end_time,f_low,f_high,species
0,Turdidae_Catharus_fuscescens_United States_201...,Training,87040,214016,1.813333,4.458667,0.000,3076.125,Turdidae_Catharus_fuscescens
1,Turdidae_Catharus_fuscescens_United States_201...,Training,350208,466944,7.296000,9.728000,26.810,3517.470,Turdidae_Catharus_fuscescens
2,Turdidae_Catharus_fuscescens_United States_201...,Training,603136,716800,12.565333,14.933333,167.245,3296.795,Turdidae_Catharus_fuscescens
3,Turdidae_Catharus_fuscescens_United States_201...,Training,921600,1029120,19.200000,21.440000,86.995,3216.545,Turdidae_Catharus_fuscescens
4,Turdidae_Catharus_fuscescens_United States_201...,Training,1159168,1264640,24.149333,26.346667,0.000,3076.120,Turdidae_Catharus_fuscescens


In [8]:
val_list_df = []

for svl_file in os.listdir(ANNOTATION_VAL):
    svl_path = os.path.join(ANNOTATION_VAL, svl_file)

    audio_file = resolve_audio_filename(svl_filename=svl_file, audio_dir=AUDIO_VAL)
    if audio_file is None:
        print(f"No audio for {svl_file}")
        continue

    df = parse_single_svl(
        svl_path=svl_path,
        audio_filename=audio_file,
        split="Validation"
    )
    val_list_df.append(df)
val_df = pd.concat(val_list_df, ignore_index=True)
val_df.head()

,audio_file,split,start_frame,end_frame,start_time,end_time,f_low,f_high,species
0,Fringillidae_Serinus_serinus_Portugal_2019-03-...,Validation,60416,246784,1.258667,5.141333,0.0000,6143.8100,Fringillidae_Serinus_serinus
1,Fringillidae_Serinus_serinus_Portugal_2019-03-...,Validation,379904,526336,7.914667,10.965333,0.0000,6457.1500,Fringillidae_Serinus_serinus
2,Fringillidae_Serinus_serinus_Portugal_2019-03-...,Validation,609280,706560,12.693333,14.720000,0.0000,6088.5200,Fringillidae_Serinus_serinus
3,Fringillidae_Serinus_serinus_Portugal_2019-03-...,Validation,852992,937984,17.770667,19.541333,0.0000,5867.3500,Fringillidae_Serinus_serinus
4,Turdidae_Catharus_fuscater_Panama_2017-04-29_X...,Validation,24576,67584,0.512000,1.408000,2457.5585,3379.1215,Turdidae_Catharus_fuscater


In [9]:
print(f"Num species: {train_df['species'].nunique()}")
sorted(train_df['species'].unique())

Num species: 22


['Fringillidae_Serinus_canicollis',
 'Fringillidae_Serinus_serinus',
 'Paridae_Hypocnemis_cantator',
 'Paridae_Hypocnemis_hypoxantha',
 'Paridae_Hypocnemis_peruviana',
 'Paridae_Hypocnemis_striata',
 'Paridae_Saxicola_gutturalis',
 'Paridae_Saxicola_rubetra',
 'Paridae_Saxicola_rubicola',
 'Paridae_Saxicola_tectes',
 'Paridae_Saxicola_torquatus',
 'Troglodytidae_Troglodytes_aedon',
 'Troglodytidae_Troglodytes_hiemalis',
 'Troglodytidae_Troglodytes_pacificus',
 'Troglodytidae_Troglodytes_troglodytes',
 'Turdidae_Catharus_aurantiirostris',
 'Turdidae_Catharus_bicknelli',
 'Turdidae_Catharus_fuscater',
 'Turdidae_Catharus_fuscescens',
 'Turdidae_Catharus_guttatus',
 'Turdidae_Catharus_minimus',
 'Turdidae_Catharus_ustulatus']

In [10]:
print(f"Num species: {val_df['species'].nunique()}")
sorted(val_df['species'].unique())

Num species: 22


['Fringillidae_Serinus_canicollis',
 'Fringillidae_Serinus_serinus',
 'Paridae_Hypocnemis_cantator',
 'Paridae_Hypocnemis_hypoxantha',
 'Paridae_Hypocnemis_peruviana',
 'Paridae_Hypocnemis_striata',
 'Paridae_Saxicola_gutturalis',
 'Paridae_Saxicola_rubetra',
 'Paridae_Saxicola_rubicola',
 'Paridae_Saxicola_tectes',
 'Paridae_Saxicola_torquatus',
 'Troglodytidae_Troglodytes_aedon',
 'Troglodytidae_Troglodytes_hiemalis',
 'Troglodytidae_Troglodytes_pacificus',
 'Troglodytidae_Troglodytes_troglodytes',
 'Turdidae_Catharus_aurantiirostris',
 'Turdidae_Catharus_bicknelli',
 'Turdidae_Catharus_fuscater',
 'Turdidae_Catharus_fuscescens',
 'Turdidae_Catharus_guttatus',
 'Turdidae_Catharus_minimus',
 'Turdidae_Catharus_ustulatus']

In [11]:
val_df, test_df = train_test_split(val_df, test_size=0.5, stratify=val_df["species"], random_state=64)
val_df = val_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

In [12]:
print("Train:", len(train_df))
print("Val:", len(val_df))
print("Test:", len(test_df))

Train: 5976
Val: 1332
Test: 1333


In [13]:
print(train_df['species'].value_counts())
print(val_df['species'].value_counts())
print(test_df['species'].value_counts())

species
Paridae_Saxicola_rubetra                 1475
Troglodytidae_Troglodytes_troglodytes     627
Troglodytidae_Troglodytes_aedon           570
Turdidae_Catharus_aurantiirostris         547
Turdidae_Catharus_guttatus                420
Paridae_Saxicola_rubicola                 411
Fringillidae_Serinus_serinus              387
Turdidae_Catharus_fuscater                225
Troglodytidae_Troglodytes_hiemalis        197
Turdidae_Catharus_ustulatus               194
Turdidae_Catharus_fuscescens              159
Paridae_Hypocnemis_peruviana              139
Troglodytidae_Troglodytes_pacificus       133
Paridae_Hypocnemis_hypoxantha             108
Paridae_Hypocnemis_cantator               106
Turdidae_Catharus_minimus                  81
Paridae_Saxicola_tectes                    48
Fringillidae_Serinus_canicollis            43
Paridae_Hypocnemis_striata                 31
Paridae_Saxicola_gutturalis                30
Turdidae_Catharus_bicknelli                27
Paridae_Saxicola_torquatus

In [14]:
train_path = os.path.join(OUTPUT_PATH, "train_annotation.csv")
val_path = os.path.join(OUTPUT_PATH, "val_annotation.csv")
test_path = os.path.join(OUTPUT_PATH, "test_annotation.csv")

train_df.to_csv(train_path, index=False)
val_df.to_csv(val_path, index=False)
test_df.to_csv(test_path, index=False)